# 🏏 CricAnalytica: ICC Men's T20 World Cup 2026 — Data Preprocessing & Insights
**Tournament:** ICC Men's T20 World Cup 2026 | **Hosts:** India & Sri Lanka | **Champion:** India 🏆



---
**Pipeline:**
1. Load JSON & CSV data
2. Clean & preprocess each dataset
3. Merge datasets
4. Export clean CSVs & JSONs ready for Power BI

In [ ]:
import pandas as pd
import json
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded successfully')

## 📋 1. Matches Summary — Load & Inspect

In [ ]:
with open('WC_t20_json_files/t20_wc_match_results.json') as f:
    data = json.load(f)
data

In [ ]:
# Convert match summary list to DataFrame
df_matches = pd.DataFrame(data['matchSummary'])
print(f'Shape: {df_matches.shape}')
df_matches.head()

### 🧹 1.1 Clean Match Data

In [ ]:
# Parse matchDate to datetime
df_matches['matchDate'] = pd.to_datetime(df_matches['matchDate'], format='%b %d, %Y')

# Extract stage from ground + context (already a column in CSV version)
# Standardize margin — replace 'N/R' with NaN
df_matches['margin'] = df_matches['margin'].replace('N/R', np.nan)

# Flag No Result matches
df_matches['no_result'] = df_matches['winner'] == 'No Result'

# Extract match number from scorecard field
df_matches['match_number'] = df_matches['scorecard'].str.extract(r'(\d+)').astype(int)

print('Null values:')
print(df_matches.isnull().sum())
df_matches.dtypes

In [ ]:
# Check for duplicate matches
dupes = df_matches.duplicated(subset=['team1','team2','matchDate'])
print(f'Duplicate match rows: {dupes.sum()}')

# Unique winners
print('\nUnique winning teams:')
print(sorted(df_matches['winner'].unique()))

In [ ]:
# Wins per team (excluding No Result)
wins = df_matches[df_matches['winner'] != 'No Result']['winner'].value_counts()
print('Wins per team (Tournament):')
print(wins)

## 🏏 2. Batting Summary — Load, Clean & Transform

In [ ]:
df_bat = pd.read_csv('WC_t20_csv_files/Batting_Summary.csv')
print(f'Shape: {df_bat.shape}')
df_bat.head(10)

In [ ]:
# Check data types and nulls
print(df_bat.dtypes)
print('\nNull values:')
print(df_bat.isnull().sum())

In [ ]:
# ---- Cleaning ----
# Ensure numeric columns are correct dtype
numeric_cols = ['innings', 'runs', 'avg', 'sr', 'hundreds', 'fifties', 'fours', 'sixes']
for col in numeric_cols:
    df_bat[col] = pd.to_numeric(df_bat[col], errors='coerce')

# Fill NaN in avg with 0 (players who were not out in all innings)
df_bat['avg'] = df_bat['avg'].fillna(0)

# ---- Derived Columns ----
# Boundary percentage: what % of runs came from boundaries
df_bat['boundary_runs'] = (df_bat['fours'] * 4) + (df_bat['sixes'] * 6)
df_bat['boundary_pct'] = ((df_bat['boundary_runs'] / df_bat['runs']) * 100).round(2)

# Tag players with centuries
df_bat['has_century'] = df_bat['hundreds'] > 0

print('Batting DataFrame after cleaning:')
df_bat.head()

In [ ]:
# ---- Key Aggregations ----

# Top 10 run scorers
print('🏏 Top 10 Run Scorers — T20 WC 2026:')
print(df_bat[['name','team','innings','runs','avg','sr']].sort_values('runs', ascending=False).head(10).to_string(index=False))

In [ ]:
# Top 10 by batting average (min 3 innings)
print('📊 Top 10 Best Batting Averages (min 3 innings):')
print(df_bat[df_bat['innings'] >= 3][['name','team','innings','runs','avg','sr']]
      .sort_values('avg', ascending=False).head(10).to_string(index=False))

In [ ]:
# Top 10 by strike rate (min 3 innings)
print('⚡ Top 10 Strike Rates (min 3 innings):')
print(df_bat[df_bat['innings'] >= 3][['name','team','innings','runs','avg','sr']]
      .sort_values('sr', ascending=False).head(10).to_string(index=False))

In [ ]:
# Most sixes
print('💥 Top 10 Six-Hitters:')
print(df_bat[['name','team','sixes','fours']].sort_values('sixes', ascending=False).head(10).to_string(index=False))

In [ ]:
# Players with centuries
print('💯 Century scorers in T20 WC 2026:')
print(df_bat[df_bat['hundreds'] > 0][['name','team','hundreds','runs','avg','sr','highest_score']]
      .sort_values('hundreds', ascending=False).to_string(index=False))

In [ ]:
# Team-wise batting totals
team_bat = df_bat.groupby('team').agg(
    total_runs=('runs', 'sum'),
    avg_sr=('sr', 'mean'),
    total_sixes=('sixes', 'sum'),
    total_fours=('fours', 'sum'),
    centuries=('hundreds', 'sum'),
    fifties=('fifties', 'sum')
).sort_values('total_runs', ascending=False).round(2)

print('\n🌍 Team-wise Batting Aggregates:')
print(team_bat.to_string())

## 🎳 3. Bowling Summary — Load, Clean & Transform

In [ ]:
df_bowl = pd.read_csv('WC_t20_csv_files/Bowling_Summary.csv')
print(f'Shape: {df_bowl.shape}')
df_bowl.head(10)

In [ ]:
print(df_bowl.dtypes)
print('\nNull values:')
print(df_bowl.isnull().sum())

In [ ]:
# ---- Cleaning ----
numeric_bowl_cols = ['innings', 'wickets', 'avg', 'economy', 'sr', 'four_wickets', 'five_wickets']
for col in numeric_bowl_cols:
    df_bowl[col] = pd.to_numeric(df_bowl[col], errors='coerce')

# Handle infinity values (e.g. bowlers with 0 wickets)
df_bowl['avg'] = df_bowl['avg'].replace([np.inf, -np.inf], np.nan)
df_bowl['sr'] = df_bowl['sr'].replace([np.inf, -np.inf], np.nan)

# Parse best bowling: split into wickets and runs
df_bowl[['bb_wickets', 'bb_runs']] = df_bowl['best_bowling'].str.split('/', expand=True).astype(float)

print('Bowling DataFrame after cleaning:')
df_bowl.head()

In [ ]:
# Top wicket takers
print('🎳 Top 10 Wicket Takers — T20 WC 2026:')
print(df_bowl[['name','team','innings','wickets','avg','economy','sr','best_bowling']]
      .sort_values('wickets', ascending=False).head(10).to_string(index=False))

In [ ]:
# Best economy (min 4 innings)
print('💨 Top 10 Most Economical Bowlers (min 4 innings):')
print(df_bowl[df_bowl['innings'] >= 4][['name','team','innings','wickets','economy','avg']]
      .sort_values('economy').head(10).to_string(index=False))

In [ ]:
# 5-wicket hauls
print('⭐ Bowlers with 5-Wicket Hauls:')
print(df_bowl[df_bowl['five_wickets'] > 0][['name','team','wickets','best_bowling','economy']].to_string(index=False))

In [ ]:
# 4-wicket hauls
print('⭐ Bowlers with 4-Wicket Hauls:')
print(df_bowl[df_bowl['four_wickets'] > 0][['name','team','wickets','four_wickets','best_bowling','economy']].to_string(index=False))

In [ ]:
# Team bowling stats
team_bowl = df_bowl.groupby('team').agg(
    total_wickets=('wickets', 'sum'),
    avg_economy=('economy', 'mean'),
    fifers=('five_wickets', 'sum'),
    four_fers=('four_wickets', 'sum')
).sort_values('total_wickets', ascending=False).round(2)

print('\n🌍 Team-wise Bowling Aggregates:')
print(team_bowl.to_string())

## 👤 4. Players Information — Load & Clean

In [ ]:
df_players = pd.read_csv('WC_t20_csv_files/Players_Information.csv')
print(f'Shape: {df_players.shape}')
df_players.head(10)

In [ ]:
# Check for nulls and types
print(df_players.isnull().sum())
print(df_players['role'].value_counts())

In [ ]:
# Clean bowlling_style: replace 'NA' with np.nan
df_players['bowling_style'] = df_players['bowling_style'].replace('NA', np.nan)

# Categorize batting hand
df_players['batting_hand'] = df_players['batting_style'].str.extract(r'^(Right|Left)')

print('Players by role:')
print(df_players['role'].value_counts())
print('\nPlayers by batting hand:')
print(df_players['batting_hand'].value_counts())

## 🔗 5. Merge Batting + Player Info

In [ ]:
df_bat_full = df_bat.merge(df_players[['name','role','batting_style','bowling_style','batting_hand','age']], 
                            on='name', how='left')
print(f'Merged batting shape: {df_bat_full.shape}')
df_bat_full.head()

In [ ]:
# Left-hand vs Right-hand run comparison
print('Runs by batting hand (top scorers):')
print(df_bat_full.groupby('batting_hand')['runs'].agg(['sum','mean','count']).round(2))

In [ ]:
# Runs by player role
print('\nRuns by player role:')
print(df_bat_full.groupby('role')['runs'].agg(['sum','mean','count']).sort_values('sum', ascending=False).round(2))

## 🔗 6. Merge Bowling + Player Info

In [ ]:
df_bowl_full = df_bowl.merge(df_players[['name','role','bowling_style','age']], 
                              on='name', how='left')
print(f'Merged bowling shape: {df_bowl_full.shape}')
df_bowl_full.head()

In [ ]:
# Wickets by bowling style
print('Wickets by bowling style:')
print(df_bowl_full.groupby('bowling_style')['wickets'].sum().sort_values(ascending=False))

## 📊 7. Match Analysis

In [ ]:
df_total = pd.read_csv('WC_t20_csv_files/Total_Matches.csv')
print(f'Total matches: {len(df_total)}')
df_total.head()

In [ ]:
# Clean Total_Matches CSV
df_total['match_date'] = pd.to_datetime(df_total['match_date'], format='%b %d %Y')

# Stage breakdown
print('Matches by stage:')
print(df_total['stage'].value_counts())

In [ ]:
# Win type distribution (by wickets vs by runs)
win_type = df_total['margin_type'].value_counts()
print('\nWin type distribution:')
print(win_type)

In [ ]:
# Matches per venue
print('\nMatches per venue:')
print(df_total['ground'].value_counts().head(10))

In [ ]:
# All India matches
india_matches = df_total[(df_total['team1'] == 'India') | (df_total['team2'] == 'India')]
print(f"India played {len(india_matches)} matches")
india_wins = india_matches[india_matches['winner'] == 'India']
print(f"India won {len(india_wins)} matches")
print(india_matches[['team1','team2','winner','margin','margin_type','ground','stage']].to_string(index=False))

## 💾 8. Export Clean Data — Ready for Power BI

In [ ]:
# Export clean CSVs
df_bat_full.to_csv('WC_t20_csv_files/Batting_Summary_clean.csv', index=False)
df_bowl_full.to_csv('WC_t20_csv_files/Bowling_Summary_clean.csv', index=False)
df_players.to_csv('WC_t20_csv_files/Players_Information_clean.csv', index=False)
df_total.to_csv('WC_t20_csv_files/Total_Matches_clean.csv', index=False)
df_matches.to_csv('WC_t20_csv_files/Match_Results_clean.csv', index=False)

# Export team aggregates
team_bat.to_csv('WC_t20_csv_files/Team_Batting_Aggregates.csv')
team_bowl.to_csv('WC_t20_csv_files/Team_Bowling_Aggregates.csv')

print('✅ All clean datasets exported to WC_t20_csv_files/')
print('\nFiles ready for Power BI:')
import os
for f in sorted(os.listdir('WC_t20_csv_files/')):
    size = os.path.getsize(f'WC_t20_csv_files/{f}')
    print(f'  📄 {f} ({size:,} bytes)')

## 🏆 9. Tournament Summary — Key Stats
A final summary of the 2026 T20 World Cup

In [ ]:
print('=' * 60)
print('   🏏 ICC Men\'s T20 World Cup 2026 — Summary Stats')
print('=' * 60)
print(f"🏆 Champion         : India (3rd title, 2nd consecutive)")
print(f"🥈 Runner-up        : New Zealand")
print(f"🏟️  Final venue       : Narendra Modi Stadium, Ahmedabad")
print(f"📅 Tournament dates : Feb 07 – Mar 08, 2026")
print(f"🌍 Co-hosts         : India & Sri Lanka")
print(f"⚽ Total matches    : 55 (across 8 venues)")
print()
print('🏏 Batting Records:')
top_bat = df_bat.sort_values('runs', ascending=False).iloc[0]
print(f"   Most runs        : {top_bat['name']} ({top_bat['team']}) — {top_bat['runs']} runs")
print(f"   Most centuries   : Sahibzada Farhan (PAK) — 2 hundreds")
print(f"   Fastest century  : Finn Allen (NZ) — 33 balls vs South Africa (SF)")
print(f"   Highest score    : Sahibzada Farhan — 110 vs Namibia")
print()
print('🎳 Bowling Records:')
top_bowl = df_bowl.sort_values('wickets', ascending=False).iloc[0]
print(f"   Most wickets     : {top_bowl['name']} ({top_bowl['team']}) — {top_bowl['wickets']} wickets")
print(f"   Best figures     : Romario Shepherd (WI) — 5/20 vs Scotland")
print(f"   Best economy     : Jasprit Bumrah (IND) — 6.21")
print()
print('🎖️  Awards:')
print(f"   Player of Tournament : Sanju Samson (India)")
print(f"   Player of Final      : Jasprit Bumrah (India) — 4/15")
print('=' * 60)

---
## ✅ Data Preprocessing Complete!

**Clean datasets ready for Power BI:**
- `Batting_Summary_clean.csv` — enriched with role, batting hand, boundary %
- `Bowling_Summary_clean.csv` — enriched with role, bowling style, best bowling split
- `Players_Information_clean.csv` — full player profiles
- `Total_Matches_clean.csv` — all 55 matches with dates
- `Match_Results_clean.csv` — from JSON, parsed dates
- `Team_Batting_Aggregates.csv` — team-level batting stats
- `Team_Bowling_Aggregates.csv` — team-level bowling stats

**Next step:** Import into Power BI → create relationships → build dashboards with DAX measures.